In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(
    r"C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/raw_data.csv", 
    low_memory=False,
    compression="gzip"
)

In [3]:
len(df)

8381556

In [4]:
df.head()

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng
0,2020-03-26 07:07:17,14626,12.313621,76.658195,12.287301,76.602280
1,2020-03-26 07:32:27,85490,12.943947,77.560745,12.954014,77.543770
2,2020-03-26 07:36:44,05408,12.899603,77.587300,12.934780,77.569950
3,2020-03-26 07:38:00,58940,12.918229,77.607544,12.968971,77.636375
4,2020-03-26 07:39:29,05408,12.899490,77.587270,12.934780,77.569950


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8381556 entries, 0 to 8381555
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   ts        str    
 1   number    str    
 2   pick_lat  float64
 3   pick_lng  float64
 4   drop_lat  float64
 5   drop_lng  float64
dtypes: float64(4), str(2)
memory usage: 383.7 MB


In [6]:
df.shape

(8381556, 6)

### A Customer_ID `number` at a particular timestamp can only have one entry
### Removing Duplicate Entries ['ts','number']

In [7]:
df[df.duplicated(subset=['ts','number'], keep=False)]

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng
235,2020-03-26 18:10:35,16795,12.967236,77.641594,13.014504,77.650856
236,2020-03-26 18:10:35,16795,12.967236,77.641594,13.014504,77.650856
407,2020-03-26 21:35:50,65856,12.917173,77.586400,12.913940,77.685280
408,2020-03-26 21:35:50,65856,12.917173,77.586400,12.913940,77.685280
443,2020-03-26 23:26:29,27554,12.933715,77.619300,12.938208,77.587520
...,...,...,...,...,...,...
8381231,2021-03-26 22:23:12,61636,12.975229,77.620370,13.017285,77.618200
8381245,2021-03-26 22:25:13,61636,12.975229,77.620370,13.017285,77.618200
8381246,2021-03-26 22:25:13,61636,12.975229,77.620370,13.017285,77.618200
8381248,2021-03-26 22:25:27,61636,12.975229,77.620370,13.017285,77.618200


### There are 113540 Duplicate Entries
#### We have 8315498 Unique timestamp, customer_id rows. 

In [8]:
## Keeping first occurence

df.drop_duplicates(subset=['ts','number'], keep ='first', inplace = True)

df.reset_index(drop = True, inplace = True)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8315498 entries, 0 to 8315497
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   ts        str    
 1   number    str    
 2   pick_lat  float64
 3   pick_lng  float64
 4   drop_lat  float64
 5   drop_lng  float64
dtypes: float64(4), str(2)
memory usage: 380.7 MB


In [10]:
df['number'] = pd.to_numeric(df['number'], errors='coerce')

#Count missing values
np.count_nonzero(df.isnull().values)

np.int64(116)

#### There are 116 NaN rows, dropping NaN rows.

In [11]:
df.dropna(inplace = True)
len(df)

8315382

In [12]:
df['number'] = pd.to_numeric(df['number'], errors='coerce', downcast='integer')
df['ts'] = pd.to_datetime(df['ts'], errors='coerce')

In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 8315382 entries, 0 to 8315497
Data columns (total 6 columns):
 #   Column    Dtype         
---  ------    -----         
 0   ts        datetime64[us]
 1   number    int32         
 2   pick_lat  float64       
 3   pick_lng  float64       
 4   drop_lat  float64       
 5   drop_lng  float64       
dtypes: datetime64[us](1), float64(4), int32(1)
memory usage: 412.4 MB


### Breaking Time to Features

In [14]:
df['mins'] = df['ts'].dt.minute
df['hour'] = df['ts'].dt.hour
df['day'] = df['ts'].dt.day 
df['month'] = df['ts'].dt.month
df['year'] = df['ts'].dt.year   
df['dayofweek'] = df['ts'].dt.dayofweek

In [15]:
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek
0,2020-03-26 07:07:17,14626,12.313621,76.658195,12.287301,76.602280,7,7,26,3,2020,3
1,2020-03-26 07:32:27,85490,12.943947,77.560745,12.954014,77.543770,32,7,26,3,2020,3
2,2020-03-26 07:36:44,5408,12.899603,77.587300,12.934780,77.569950,36,7,26,3,2020,3
3,2020-03-26 07:38:00,58940,12.918229,77.607544,12.968971,77.636375,38,7,26,3,2020,3
4,2020-03-26 07:39:29,5408,12.899490,77.587270,12.934780,77.569950,39,7,26,3,2020,3
...,...,...,...,...,...,...,...,...,...,...,...,...
8315493,2021-03-26 23:55:24,50410,12.907856,77.557870,12.954270,77.530785,55,23,26,3,2021,4
8315494,2021-03-26 23:58:15,12580,12.981010,77.694450,12.969070,77.704280,58,23,26,3,2021,4
8315495,2021-03-26 22:11:20,72339,12.924252,77.650520,12.905820,77.630570,11,22,26,3,2021,4
8315496,2021-03-26 22:12:30,72339,12.924252,77.650520,12.905820,77.630570,12,22,26,3,2021,4


In [16]:
df.to_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/preprocessed/preprocessed_1.csv',index = False, compression = 'gzip')

In [17]:
from geopy.distance import geodesic
from geopy.geocoders import Nominatim
import time
geolocator = Nominatim(user_agent="OLABikes")

## Data Cleaning with Business Understanding

### There can be cases when a user requests a ride, and their booking request is logged in our database but this user re-books his/her ride due to longer wait hours or driver refused booking or user by mistake added wrong pickup or drop locations. 

<hr>

### `Handle Case1 Rebooking Again to Same Location`: Keep only one request of same user to same pickup latitude longitude in 1hour time frame of first ride request.

<hr>

### `Handle Case2 Location entry mistake`: Keep only last request of user within 8mintues of first booking request.
#### A person booking a ride would generally book a ride that would take 8mins of bike ride time. 
#### Also, Calculate distance b/w pickup and drop. Based on distance and request time different remove bad data entries.

#### `Handle Case2.1`: Pick Up and Drop Lat-Long Distance less than 50meters = 0.05 kms; No user would like to ride for just 50meters trip. 

<hr>

### `Handle Case3`: Booking Location Outside operation zone of OLABikes

In [18]:
## Check lat-long bounding box coordinates

df['ts'] = pd.to_datetime(df['ts'])
df.sort_values(by = ['number','ts'], inplace = True)
df.reset_index(drop=True, inplace=True)

In [19]:
# you need convert first to numpy array by values and cast to int64 - output is in nanosecond, so need divide by 10 ** 9

df['booking_timestamp'] = df['ts'].astype('int64') // 10**9

In [20]:
df.head(20)

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5,1602315
1,2020-10-11 08:23:42,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
2,2020-10-11 08:23:50,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
3,2020-10-11 08:23:51,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
4,2020-10-11 08:23:54,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
5,2020-10-11 08:23:56,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
6,2020-10-11 11:57:17,-1,12.960213,77.587460,12.930824,77.609610,57,11,11,10,2020,6,1602417
7,2020-10-11 11:57:31,-1,12.960213,77.587460,12.930824,77.609610,57,11,11,10,2020,6,1602417
8,2020-10-16 17:51:07,-1,12.924353,77.549410,12.932216,77.581825,51,17,16,10,2020,4,1602870
9,2020-10-16 17:51:25,-1,12.924353,77.549410,12.932216,77.581825,51,17,16,10,2020,4,1602870


In [21]:
df['shift_booking_ts'] = df.groupby('number')['booking_timestamp'].shift(1)
df['shift_booking_ts'] = df['shift_booking_ts'].fillna(0)

In [22]:
df['shift_booking_ts'] = df['shift_booking_ts'].astype('Int64')

In [23]:
df['booking_time_diff_hr'] = round((df['booking_timestamp'] - df['shift_booking_ts'])//3600)
df['booking_time_diff_min'] = round((df['booking_timestamp'] - df['shift_booking_ts'])//60)

In [24]:
##Booking time different in mins
df['booking_time_diff_min'].value_counts().to_dict()

{np.int64(0): 6138862,
 np.int64(1): 681469,
 np.int64(2): 290695,
 np.int64(4): 148657,
 np.int64(3): 130265,
 np.int64(5): 108979,
 np.int64(7): 66255,
 np.int64(8): 59919,
 np.int64(6): 52265,
 np.int64(10): 41527,
 np.int64(9): 34799,
 np.int64(11): 33833,
 np.int64(12): 24190,
 np.int64(14): 22861,
 np.int64(15): 20103,
 np.int64(13): 18722,
 np.int64(17): 17362,
 np.int64(18): 16798,
 np.int64(20): 14970,
 np.int64(21): 12639,
 np.int64(16): 12575,
 np.int64(19): 10571,
 np.int64(24): 10017,
 np.int64(23): 8740,
 np.int64(22): 8395,
 np.int64(25): 8352,
 np.int64(27): 8298,
 np.int64(30): 7707,
 np.int64(28): 7647,
 np.int64(31): 6458,
 np.int64(26): 6239,
 np.int64(34): 5413,
 np.int64(29): 5169,
 np.int64(33): 5055,
 np.int64(37): 4774,
 np.int64(40): 4703,
 np.int64(38): 4311,
 np.int64(32): 4202,
 np.int64(35): 4189,
 np.int64(41): 3976,
 np.int64(36): 3931,
 np.int64(44): 3457,
 np.int64(43): 3366,
 np.int64(39): 3143,
 np.int64(50): 3105,
 np.int64(47): 3066,
 np.int64(2642

In [25]:
##Booking time different in hours
df['booking_time_diff_hr'].value_counts().to_dict()

{np.int64(0): 8122715,
 np.int64(1): 55991,
 np.int64(440): 23791,
 np.int64(2): 19758,
 np.int64(441): 14077,
 np.int64(442): 11594,
 np.int64(443): 11055,
 np.int64(3): 10619,
 np.int64(444): 9319,
 np.int64(447): 7360,
 np.int64(445): 6195,
 np.int64(446): 5964,
 np.int64(4): 5906,
 np.int64(448): 4654,
 np.int64(5): 3314,
 np.int64(6): 1867,
 np.int64(7): 808,
 np.int64(449): 265,
 np.int64(8): 130}

### We observe that there are 8315382 - 3248768 = 50,66,614 booking that happen in less than 1 hour of request by a user

In [26]:
len(df)

8315382

In [27]:
### Handling Case 1: Re-booking Again to Same Location within 1hour by same user

df = df[~((df.duplicated(subset=['number','pick_lat','pick_lng'],keep=False)) & (df.booking_time_diff_hr<=1))]

In [28]:
## Before removing Row Count
len(df)

3248768

###  Removed 5066614 rows in `Case1` we now have 3248768

In [29]:
df.to_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/preprocessed/preprocessed_2.csv',index = False, compression = 'gzip')

### Handling Case2: One user Books rides are different lat-long within 8mins time (ride time + driver arrival time)
#### Fraud User
#### Human error booking

In [30]:
df = pd.read_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/preprocessed/preprocessed_2.csv', compression = 'gzip')

In [31]:
df.shape

(3248768, 16)

In [32]:
print("Number of rides booked by same customer within 8mins time: {}".format(len(df[(df.booking_time_diff_min<8)])))
df = df[(df.booking_time_diff_min>=8)]

Number of rides booked by same customer within 8mins time: 2697418


### Assuming earth as ellipsoids, calculating geodesic distance b/w pickup and drop latitude and longitude

The geodesic distance is the shortest distance on the surface of an ellipsoidal model of the earth.

In [33]:
%%time
def geodestic_distance(pick_lat, pick_lng, drop_lat, drop_lng):
    # 1mile = 1.60934 Kms
    return round(geodesic((pick_lat, pick_lng), (drop_lat, drop_lng)).miles*1.60934,2)

df['geodesic_distance'] = np.vectorize(geodestic_distance)(df['pick_lat'],df['pick_lng'],df['drop_lat'],df['drop_lng'])

CPU times: total: 32.1 s
Wall time: 32.4 s


##### Number of rides booked but same customer within 8mins time: 2697418

In [34]:
df[df['geodesic_distance']<=0.5]['geodesic_distance'].value_counts()

geodesic_distance
0.00    2192
0.01     551
0.02     360
0.03     248
0.50     206
0.48     175
0.04     173
0.49     170
0.05     161
0.45     157
0.39     156
0.46     151
0.47     139
0.42     138
0.43     134
0.44     133
0.06     126
0.40     124
0.07     124
0.35     118
0.41     115
0.36     113
0.09     111
0.12     110
0.38     105
0.30     105
0.34     104
0.37     104
0.33     102
0.26      96
0.14      96
0.10      96
0.08      94
0.31      93
0.32      93
0.24      91
0.17      87
0.27      85
0.28      84
0.13      83
0.22      81
0.29      81
0.11      79
0.23      77
0.21      77
0.18      75
0.15      70
0.20      70
0.25      64
0.19      63
0.16      56
Name: count, dtype: int64

### Handle Case 2.1: Removing ride request less than 0.05 miles = 50meters

In [35]:
print("Number of Rides Requests less than 50meters: {}".format(len(df[df.geodesic_distance<=0.05])))

Number of Rides Requests less than 50meters: 3685


In [36]:
df = df[df.geodesic_distance>0.05]
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp,shift_booking_ts,booking_time_diff_hr,booking_time_diff_min,geodesic_distance
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5,1602315,0,445,26705,17.38
1,2020-10-30 09:00:44,-1,12.945731,77.622500,12.973030,77.616840,0,9,30,10,2020,4,1604048,1602870,0,19,3.08
2,2020-11-27 20:16:20,-1,13.031985,77.571280,12.960211,77.646910,16,20,27,11,2020,4,1606508,1604048,0,41,11.42
4,2020-12-05 22:09:55,-1,12.938531,77.578940,12.952660,77.568825,9,22,5,12,2020,5,1607206,1606509,0,11,1.91
8,2020-12-17 17:40:20,-1,12.979331,77.577910,12.944552,77.596510,40,17,17,12,2020,3,1608226,1607418,0,13,4.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3248756,2020-11-23 12:51:56,99999,13.042774,77.609170,13.059187,77.596280,51,12,23,11,2020,0,1606135,1602266,1,64,2.29
3248758,2021-01-11 17:45:09,99999,13.059011,77.596270,13.042389,77.611790,45,17,11,1,2021,0,1610387,1606814,0,59,2.49
3248761,2021-02-07 09:10:15,99999,12.956661,77.521935,12.925382,77.686516,10,9,7,2,2021,6,1612689,1610568,0,35,18.19
3248764,2021-02-19 20:43:25,99999,13.029296,77.592580,12.927923,77.627106,43,20,19,2,2021,4,1613767,1613158,0,10,11.82


In [37]:
len(df)

547665

In [38]:
df.to_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/preprocessed/preprocessed_3.csv',index = False, compression = 'gzip')

In [39]:
import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

geolocator = Nominatim(user_agent="my_app", timeout=20)

places = ["India", "Bengaluru, Karnataka, India", "Karnataka, India"]

for place in places:
    try:
        loc = geolocator.geocode(place, exactly_one=True, addressdetails=True)
        if loc:
            print(place, loc.raw.get("boundingbox"))
        else:
            print(place, "No result")
    except (GeocoderTimedOut, GeocoderUnavailable) as e:
        print(place, f"Failed: {e}")
    time.sleep(1)

India ['6.5531169', '35.6729307', '67.9544415', '97.3950905']
Bengaluru, Karnataka, India ['12.8334905', '13.1426196', '77.4598797', '77.7840639']
Karnataka, India ['11.5945587', '18.4766494', '74.0543908', '78.5875761']


### Handle Case3: Rides request in non-operational regions
OLA Bikes OPERATION CITY (Bangalore)

### Ride requests due to some bug or crash in app.
<hr>

#### India: 'boundingbox': ['6.5531169', '35.6729307', '67.9544415', '97.3950905']
#### Bangalore:'boundingbox': ['12.8334905', '13.1426196', '77.4598797', '77.7840639']
#### Karnataka: 'boundingbox': ['11.5945587', '18.4766494', '74.0543908', '78.5875761']

In [40]:
df = pd.read_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/preprocessed/preprocessed_3.csv', compression = 'gzip')
location = geolocator.geocode("India")
location.raw

{'place_id': 250717345,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'relation',
 'osm_id': 304716,
 'lat': '22.3511148',
 'lon': '78.6677428',
 'class': 'boundary',
 'type': 'administrative',
 'place_rank': 4,
 'importance': 0.8910953652546315,
 'addresstype': 'country',
 'name': 'India',
 'display_name': 'India',
 'boundingbox': ['6.5531169', '35.6729307', '67.9544415', '97.3950905']}

In [41]:
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp,shift_booking_ts,booking_time_diff_hr,booking_time_diff_min,geodesic_distance
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5,1602315,0,445,26705,17.38
1,2020-10-30 09:00:44,-1,12.945731,77.622500,12.973030,77.616840,0,9,30,10,2020,4,1604048,1602870,0,19,3.08
2,2020-11-27 20:16:20,-1,13.031985,77.571280,12.960211,77.646910,16,20,27,11,2020,4,1606508,1604048,0,41,11.42
3,2020-12-05 22:09:55,-1,12.938531,77.578940,12.952660,77.568825,9,22,5,12,2020,5,1607206,1606509,0,11,1.91
4,2020-12-17 17:40:20,-1,12.979331,77.577910,12.944552,77.596510,40,17,17,12,2020,3,1608226,1607418,0,13,4.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547660,2020-11-23 12:51:56,99999,13.042774,77.609170,13.059187,77.596280,51,12,23,11,2020,0,1606135,1602266,1,64,2.29
547661,2021-01-11 17:45:09,99999,13.059011,77.596270,13.042389,77.611790,45,17,11,1,2021,0,1610387,1606814,0,59,2.49
547662,2021-02-07 09:10:15,99999,12.956661,77.521935,12.925382,77.686516,10,9,7,2,2021,6,1612689,1610568,0,35,18.19
547663,2021-02-19 20:43:25,99999,13.029296,77.592580,12.927923,77.627106,43,20,19,2,2021,4,1613767,1613158,0,10,11.82


In [42]:
## How many rides outside india?
df[(df.pick_lat<=6.5531169) | (df.pick_lat>=35.6729307) | (df.pick_lng<=67.9544415) | (df.pick_lng>=97.3950905) | (df.pick_lat<=6.5531169) | (df.pick_lat>=35.6729307) | (df.pick_lng<=67.9544415) | (df.pick_lng>=97.3950905)]

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp,shift_booking_ts,booking_time_diff_hr,booking_time_diff_min,geodesic_distance
23994,2020-08-30 18:40:45,4133,6.525380,38.829945,12.983918,77.538660,40,18,30,8,2020,6,1598812,1597404,0,23,4301.97
38696,2021-01-03 21:07:04,6783,67.696650,-0.658846,13.032407,77.604744,7,21,3,1,2021,6,1609708,1609076,0,10,8176.72
91252,2020-12-30 10:29:19,16181,0.000000,0.000000,12.828831,77.588000,29,10,30,12,2020,2,1609324,1608143,0,19,8670.97
189599,2020-12-29 12:24:17,33876,0.000000,0.000000,12.989280,77.619330,24,12,29,12,2020,1,1609244,1608328,0,15,8675.22
195746,2020-12-27 09:23:26,34958,0.000000,0.000000,12.934451,77.611320,23,9,27,12,2020,6,1609061,1593632,4,257,8674.06
199809,2020-11-24 18:27:12,35688,0.000000,0.000000,12.946287,77.580060,27,18,24,11,2020,1,1606242,0,446,26770,8670.73
407129,2020-11-23 07:55:26,73993,0.000000,0.000000,20.355660,85.815950,55,7,23,11,2020,0,1606118,0,446,26768,9579.89
471181,2020-04-26 18:11:48,85865,28.375704,19.159657,13.022936,77.551570,11,18,26,4,2020,6,1587924,1586205,0,28,6255.52


In [43]:
df_outside_india = df[
    (df.pick_lat <= 6.2325274) |
    (df.pick_lat >= 35.6745457) |
    (df.pick_lng <= 68.1113787) |
    (df.pick_lng >= 97.395561) |
    (df.drop_lat <= 6.2325274) |
    (df.drop_lat >= 35.6745457) |
    (df.drop_lng <= 68.1113787) |
    (df.drop_lng >= 97.395561)
]

df_outside_india.shape

(146, 17)

### OLA Bikes is only operational in India
### Removing all rides for which pickup or drop is outside INDIA.
#### Number of such cases: 146

In [44]:
df.reset_index(inplace = True, drop = True)
df_outside_india = df[
    (df.pick_lat <= 6.2325274) |
    (df.pick_lat >= 35.6745457) |
    (df.pick_lng <= 68.1113787) |
    (df.pick_lng >= 97.395561) |
    (df.drop_lat <= 6.2325274) |
    (df.drop_lat >= 35.6745457) |
    (df.drop_lng <= 68.1113787) |
    (df.drop_lng >= 97.395561)
]
df = df[~df.index.isin(df_outside_india.index)].reset_index(drop = True)

In [45]:
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp,shift_booking_ts,booking_time_diff_hr,booking_time_diff_min,geodesic_distance
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5,1602315,0,445,26705,17.38
1,2020-10-30 09:00:44,-1,12.945731,77.622500,12.973030,77.616840,0,9,30,10,2020,4,1604048,1602870,0,19,3.08
2,2020-11-27 20:16:20,-1,13.031985,77.571280,12.960211,77.646910,16,20,27,11,2020,4,1606508,1604048,0,41,11.42
3,2020-12-05 22:09:55,-1,12.938531,77.578940,12.952660,77.568825,9,22,5,12,2020,5,1607206,1606509,0,11,1.91
4,2020-12-17 17:40:20,-1,12.979331,77.577910,12.944552,77.596510,40,17,17,12,2020,3,1608226,1607418,0,13,4.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547514,2020-11-23 12:51:56,99999,13.042774,77.609170,13.059187,77.596280,51,12,23,11,2020,0,1606135,1602266,1,64,2.29
547515,2021-01-11 17:45:09,99999,13.059011,77.596270,13.042389,77.611790,45,17,11,1,2021,0,1610387,1606814,0,59,2.49
547516,2021-02-07 09:10:15,99999,12.956661,77.521935,12.925382,77.686516,10,9,7,2,2021,6,1612689,1610568,0,35,18.19
547517,2021-02-19 20:43:25,99999,13.029296,77.592580,12.927923,77.627106,43,20,19,2,2021,4,1613767,1613158,0,10,11.82


In [46]:
print("Number of Good Ride Requests: {}".format(len(df)))

Number of Good Ride Requests: 547519


In [47]:
## How many pickups and drops are outside bangalore?
# ['12.8334905', '13.1426196', '77.4598797', '77.7840639']
pck_outside_bng = df[(df.pick_lat<=12.8334905) | (df.pick_lat>=13.1426196) | (df.pick_lng<=77.4598797) | (df.pick_lng>=77.7840639)]
drp_outside_bng = df[(df.drop_lat<=12.8334905) | (df.drop_lat>=13.1426196) | (df.drop_lng<=77.4598797) | (df.drop_lng>=77.7840639)]
print("Number of Pickup Requests Outside Bangalore: ",len(pck_outside_bng))
print("Number of Customers pickup outside Bangalore: ",len(np.unique(pck_outside_bng['number'].values)))

print("Number of Drops Requests Outside Bangalore: ",len(drp_outside_bng))
print("Number of Customers Drop outside Bangalore: ",len(np.unique(drp_outside_bng['number'].values)))

Number of Pickup Requests Outside Bangalore:  30930
Number of Customers pickup outside Bangalore:  14309
Number of Drops Requests Outside Bangalore:  34321
Number of Customers Drop outside Bangalore:  17079


In [48]:
### Bounding PickUp Lat-Long Within State Karnataka
# ['11.5945587', '18.4766494', '74.0543908', '78.5875761']
pck_outside_KA = df[(df.pick_lat<=11.5945587) | (df.pick_lat>=18.4766494) | (df.pick_lng<=74.0543908) | (df.pick_lng>=78.5875761)]
drp_outside_KA = df[(df.drop_lat<=11.5945587) | (df.drop_lat>=18.4766494) | (df.drop_lng<=74.0543908) | (df.drop_lng>=78.5875761)]
print("Pickups Outisde KA: {} \nDrop outside KA: {}".format(len(pck_outside_KA),len(drp_outside_KA)))
print("Number of Customers Drop outside KA: ",len(np.unique(drp_outside_KA['number'].values)))
print("Number of Customers pickup outside KA: ",len(np.unique(pck_outside_KA['number'].values)))

Pickups Outisde KA: 6773 
Drop outside KA: 7039
Number of Customers Drop outside KA:  4278
Number of Customers pickup outside KA:  4028


In [49]:
total_ride_outside_KA = df[(df.pick_lat<=11.5945587) | (df.pick_lat>=18.4766494) | (df.pick_lng<=74.0543908) | (df.pick_lng>=78.5875761) | (df.drop_lat<=11.5945587) | (df.drop_lat>=18.4766494) | (df.drop_lng<=74.0543908) | (df.drop_lng>=78.5875761)]

In [50]:
print("Total Ride Outside Karnataka: {}".format(len(total_ride_outside_KA)))

Total Ride Outside Karnataka: 7048


### Total Ride Outside Karnataka: 7048
### OLA Bikes doesnot provide intercity requests. Considering these as system error requests

In [51]:
## Rides for which geodesic distance > 500kms
## Pickup and drop not of KA (state where we have maximum booking requests and user base)
suspected_bad_rides = total_ride_outside_KA[total_ride_outside_KA.geodesic_distance > 500]
suspected_bad_rides

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp,shift_booking_ts,booking_time_diff_hr,booking_time_diff_min,geodesic_distance
6848,2020-12-31 18:24:14,1079,23.059650,72.592384,12.985226,77.567955,24,18,31,12,2020,3,1609439,1603531,1,98,1232.88
8515,2020-06-05 16:41:41,1378,12.910915,77.610085,19.766151,74.477840,41,16,5,6,2020,4,1591375,1586805,1,76,829.07
9553,2020-07-21 07:06:21,1566,12.893449,77.623940,21.195150,72.795494,6,7,21,7,2020,1,1595315,1594216,0,18,1052.50
12435,2020-04-27 22:00:45,2074,12.948681,77.648940,30.316494,78.032190,0,22,27,4,2020,0,1588024,0,441,26467,1923.56
15676,2020-06-05 20:27:56,2643,12.977254,77.549220,28.552588,77.246270,27,20,5,6,2020,4,1591388,0,442,26523,1724.77
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542361,2020-06-17 12:28:40,99158,12.963843,77.577484,26.842180,80.938576,28,12,17,6,2020,2,1592396,1591445,0,15,1575.95
543071,2020-07-11 12:47:54,99271,12.908705,77.586120,18.530767,73.848020,47,12,11,7,2020,5,1594471,1586799,2,127,739.86
543779,2020-06-24 16:34:35,99395,12.972305,77.614930,28.979435,77.689580,34,16,24,6,2020,2,1593016,0,442,26550,1772.36
543780,2020-07-29 15:29:45,99395,12.975827,77.605644,28.979435,77.689580,29,15,29,7,2020,2,1596036,1593084,0,49,1771.97


### There are 170 rides which are >500kms geodesic distance and are pickup & drop outside KA, these are suspected rides. 

In [52]:
df = df[~df.index.isin(suspected_bad_rides.index)].reset_index(drop = True)

In [53]:
print("Number of Good Ride Requests: {}".format(len(df)))

Number of Good Ride Requests: 547349


In [54]:
dataset = df[['ts', 'number', 'pick_lat','pick_lng','drop_lat','drop_lng','geodesic_distance','hour','mins','day','month','year','dayofweek','booking_timestamp','booking_time_diff_hr', 'booking_time_diff_min']]

In [55]:
dataset.to_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/clean_data.csv',index = False, compression = 'gzip')

## Statistic

### Ride request of same user with same timestamp: 113540
### None user_id: 116
### Number of requests to same pickup lat-long by a user within 1hour: 5,066,614
### Number of rides by a user within 8 mins of booking to different pickup lat-long: 26,97,418
### Number of Rides Requests less than 50meters of pickup and drop: 3,685
### Number of Rides pickup or drop lat-long outside India: 146

### Our majority ride state is from Karnataka (Bangalore)
### Total Ride Outside Karnataka (pickup or drop): 7,048
### Rides which are outside KA and pickup to drop distance is >500kms: 170

## Number of Good Ride Requests: 5,47,349